# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [27]:
# Imports — libraries used across the app

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr  # web UI for the chat app
import json  # parse tool arguments from the model
from datetime import datetime, timezone
import tempfile  # save TTS audio to a temp file for Gradio

In [28]:
# Model names — same as Week 1 exercise

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [29]:
# Setup — API clients and tutor personality (from Week 1)

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Missing OPENAI_API_KEY. Add it to a .env file in the project root "
        "(see week1/day1.ipynb and the troubleshooting notebook if needed)."
    )

openai_client = OpenAI()

# Ollama runs locally — needs `ollama serve` and `ollama pull llama3.2`
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Sent with every chat so the model stays a technical tutor
SYSTEM_PROMPT = """You are a patient technical tutor. Explain clearly and accurately.
Use short sections or bullet points when helpful. If you show code, keep it minimal. If the user asks for the current time or a calculation, use the available tools."""


In [30]:
# Tool 1 — real clock time (GPT can call this when user asks "what time is it?")

def get_current_time(timezone_name="UTC"):
    from zoneinfo import ZoneInfo
    try:
        tz = ZoneInfo(timezone_name)
    except Exception:
        tz = timezone.utc  # fall back if timezone name is invalid
    now = datetime.now(tz)
    return now.strftime(f"Current time in {timezone_name}: %Y-%m-%d %H:%M:%S %Z")

In [31]:
# Tool 2 — simple calculator (GPT can call this for math questions)

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "Invalid expression"
    return str(eval(expression))  # ok for a learning demo; not for production

In [32]:
# Tool schemas — tells GPT what tools exist and how to call them (Day 4 pattern)

get_current_time_tool = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a timezone.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone_name": {
                    "type": "string",
                    "description": "IANA timezone, e.g. UTC or Europe/London",
                }
            },
            "required": [],
            "additionalProperties": False,
        },
    },
}

calculate_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a math expression, e.g. 17 * 23",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression to evaluate",
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
}

TOOLS = [get_current_time_tool, calculate_tool]

In [33]:
# Run whichever tool GPT asked for, then send results back to the model

def handle_tool_calls(message):
    results = []
    for tc in message.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments or "{}")
        if name == "get_current_time":
            out = get_current_time(**args)
        elif name == "calculate":
            out = calculate(**args)
        else:
            out = f"Unknown tool: {name}"
        print(f"Tool called: {name}({args})", flush=True)
        results.append({"role": "tool", "content": out, "tool_call_id": tc.id})
    return results

In [34]:
# Audio helpers (Day 5) — always use OpenAI, even when chat model is Llama

def transcribe_audio(audio_path):
    """Mic recording → text via Whisper."""
    if not audio_path:
        return ""
    with open(audio_path, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    print(f"Transcribed: {transcript.text}", flush=True)
    return transcript.text


def text_to_speech(text):
    """Answer text → spoken audio bytes."""
    if not text:
        return None
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text[:4096],  # trim very long answers for TTS
    )
    return response.content


def save_audio_bytes(audio_bytes):
    """Gradio needs a file path to play audio."""
    f = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    f.write(audio_bytes)
    f.close()
    return f.name

In [35]:
# Core chat logic — pick model, call API, stream answer back to Gradio

def chat_stream(message, history, model_choice):
    # Route to the right client based on dropdown
    if model_choice == "Llama 3.2":
        client = ollama_client
        model = MODEL_LLAMA
    else:
        client = openai_client
        model = MODEL_GPT

    print(f"Calling model: {model} (selected: {model_choice})", flush=True)

    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Llama path — stream tokens one by one (no tools)
    if model_choice == "Llama 3.2":
        stream = client.chat.completions.create(
            model=model, messages=messages, stream=True
        )
        response_text = ""
        for chunk in stream:
            response_text += chunk.choices[0].delta.content or ""
            yield response_text  # Gradio updates the chat as text grows
        return

    # GPT path — may call tools first, then return final answer
    response = client.chat.completions.create(
        model=model, messages=messages, tools=TOOLS
    )
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        messages.extend(handle_tool_calls(msg))
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS
        )

    final = response.choices[0].message.content or ""
    yield final

In [ ]:
# Gradio callback — ties together text, mic, model choice, chat, and TTS

def respond(message, history, model_choice, audio_file, speak_reply):
    history = history or []

    # If user recorded audio, turn it into text first
    if audio_file:
        message = transcribe_audio(audio_file) or message
    if not message or not message.strip():
        return "", history, None

    history = history + [{"role": "user", "content": message.strip()}]
    full = ""

    # Stream model response into the chat UI
    for chunk in chat_stream(message.strip(), history[:-1], model_choice):
        full = chunk
        yield "", history + [{"role": "assistant", "content": full}], None

    # Optionally speak the final answer aloud
    audio_path = None
    if speak_reply and full:
        audio_bytes = text_to_speech(full)
        if audio_bytes:
            audio_path = save_audio_bytes(audio_bytes)
    yield "", history + [{"role": "assistant", "content": full}], audio_path


# --- Gradio UI (run cells 1–9 first) ---

with gr.Blocks(title="Technical Tutor") as demo:
    gr.Markdown(
        "## Technical Q&A — Week 2\n"
        "*Week 1 technical tutor upgraded with Gradio, model switching, tools, and audio.*"
    )

    with gr.Row():
        model_dropdown = gr.Dropdown(
            ["GPT-4o-mini", "Llama 3.2"],
            value="GPT-4o-mini",
            label="Model",
        )
        speak_reply = gr.Checkbox(value=True, label="Speak reply")

    chatbot = gr.Chatbot(height=450, type="messages")
    audio_out = gr.Audio(label="Assistant voice", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask a technical question...",
            scale=4,
            label="Message",
        )
        send = gr.Button("Send", variant="primary", scale=1)
        clear = gr.Button("Clear", scale=1)

    audio_in = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Or speak your question",
    )

    # One-click test questions (includes Week 1 example)
    gr.Examples(
        examples=[
            [
                'Please explain what this code does and why:\n'
                'yield from {book.get("author") for book in books if book.get("author")}'
            ],
            ["What time is it in UTC?"],
            ["Calculate 256 / 8"],
        ],
        inputs=msg,
    )

    inputs = [msg, chatbot, model_dropdown, audio_in, speak_reply]
    outputs = [msg, chatbot, audio_out]

    msg.submit(respond, inputs=inputs, outputs=outputs)
    send.click(respond, inputs=inputs, outputs=outputs)
    clear.click(lambda: ([], None), outputs=[chatbot, audio_out])

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Calling model: gpt-4o-mini (selected: GPT-4o-mini)
Tool called: calculate({'expression': '256 / 8'})
Calling model: gpt-4o-mini (selected: GPT-4o-mini)
Calling model: llama3.2 (selected: Llama 3.2)
